In [ ]:
#-- R libraries
library(dplyr)
library(data.table)
library(tidyr)
library(stringr)
library(lubridate)

In [ ]:
#-- Read into R
PrescriptionData <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/AD_processed.csv") %>%
  mutate(drug_exposure_start_date = as.Date(drug_exposure_start_date),
         final_end_date = as.Date(final_end_date))

pheno <- read.csv(file.path("/home/jupyter/workspaces/geneticsofantidepressantresponse/data", "Phenotypes.csv"))

In [ ]:
#=== Implementing study inclusion criteria ===
#-- Filter for Antidepressants
PrescriptionData_AD <- PrescriptionData %>%
  filter(!Category %in% c("Atypical Antipsychotic", "Mood_Stabilizer", "Typical Antipsychotic", 
                          "Serotonin_Precursor", "AminoAcid", "Augmentation"))
dim(PrescriptionData_AD)
length(unique(PrescriptionData_AD$person_id))

#-- Date of first recorded antidepressant dispense for all individuals
first_dispense <- PrescriptionData_AD %>%
  group_by(person_id) %>%
  summarise(
    DateFirstDispense = min(drug_exposure_start_date)
  )

#-- Filter for those at least 12-years old at first AD dispense
input_data <- PrescriptionData %>%
    left_join(first_dispense, by = "person_id") %>%
    left_join(pheno, by = "person_id") %>%
    mutate(
      age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
    ) %>%
    filter(age_at_first_AD >= 12)

PrescriptionData <- PrescriptionData %>%
    filter(person_id %in% input_data$person_id)

#-- Antidepressant prescriptions
PrescriptionData_AD <- PrescriptionData %>%
  filter(!Category %in% c("Atypical Antipsychotic", "Mood_Stabilizer", "Typical Antipsychotic", 
                          "Serotonin_Precursor", "AminoAcid", "Augmentation")) 
dim(PrescriptionData_AD)
length(unique(PrescriptionData_AD$person_id))

In [ ]:
#-- Study period
min(PrescriptionData_AD$drug_exposure_start_date); max(PrescriptionData_AD$drug_exposure_start_date)

In [ ]:
#-- Calculate Prescription Episode duration and label as 'continuation' or 'discontinuation' for all drugs
TreatmentContinuation=90

PrescriptionEpisodes_all <- PrescriptionData %>%
    group_by(person_id, drug, PrescriptionEpisode) %>%
    summarize(
      FirstDispense = min(drug_exposure_start_date),
      ExpectedEndDate = max(final_end_date),
      EpisodeDuration = as.numeric(difftime(max(final_end_date),  min(drug_exposure_start_date), units = "days")),
      EpisodeOutcome = ifelse(as.numeric(difftime(max(final_end_date), min(drug_exposure_start_date), units = "days")) >= TreatmentContinuation, 
                              "Continuation", "Discontinuation"),
      .groups = "drop"
    )

#-- Identify atypical antipsychotic and antidepressant prescription episodes (of any length)
AAP_PrescriptionEpisodes <- PrescriptionEpisodes_all %>%
  filter(
    str_detect(
      drug,
      "Atypical Antipsychotic|Mood_Stabilizer"
    )
  )
AD_PrescriptionEpisodes <- PrescriptionEpisodes_all %>%
  filter(
    !str_detect(
      drug,
      "Atypical Antipsychotic|Mood_Stabilizer|Typical Antipsychotic|Serotonin_Precursor|AminoAcid|Augmentation"
    )
  )

In [ ]:
#-- Rank of antidepressant prescription data
rank_prescriptions <- PrescriptionData_AD %>%
    select(person_id, drug, drug_exposure_start_date) %>%
    group_by(year = year(drug_exposure_start_date), drug) %>%
    summarise(
    TotalPrescriptions = n()
    ) %>%
    group_by(year) %>%
    mutate(rank = as.integer(min_rank(-TotalPrescriptions))) %>%
    arrange(year, rank)

#-- Save locally first
write.csv(rank_prescriptions, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Rank_prescriptions.csv", 
          row.names = FALSE)

In [ ]:
#== Calculate general adherence scores using only antidepressant prescription data

#-- Check whether a prescription is adherent or not
AdherenceWindowUpper = 90

#-- Adherence across all antidepressants
PrescriptionData_adherence <- PrescriptionData_AD %>%
  group_by(person_id) %>%
  mutate(
    PrevExpectedEndDate = lag(final_end_date),
    Adherent = ifelse(
      !is.na(PrevExpectedEndDate) & drug_exposure_start_date <= (PrevExpectedEndDate + AdherenceWindowUpper), 1, 0
    )
  ) %>%
  ungroup()

#-- Adherence within antidepressants
ATC_PrescriptionData_adherence <- PrescriptionData_AD %>%
  group_by(person_id, drug) %>%
  mutate(
    PrevExpectedEndDate = lag(final_end_date),
    Adherent = ifelse(
      !is.na(PrevExpectedEndDate) & drug_exposure_start_date <= (PrevExpectedEndDate + AdherenceWindowUpper), 1, 0
    )
  ) %>%
  ungroup()

#-- Mean general prescription adherence scores
Adherence <- PrescriptionData_adherence %>%
  mutate(Adherent = replace_na(Adherent, 0)) %>%
  group_by(person_id) %>%
  summarise(
    General_adherence_score = mean(Adherent)
  )
ATC_Adherence <- PrescriptionData_adherence %>%
  mutate(Adherent = replace_na(Adherent, 0)) %>%
  group_by(person_id, drug) %>%
  summarise(
    ATC_Adherence_score = mean(Adherent)
  )

#-- Combine information
info <- first_dispense %>%
  full_join(Adherence, by = "person_id") %>%
  full_join(ATC_Adherence, by = "person_id")

#-- Save locally first
write.csv(info, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/AD_dispense_characteristics.csv", row.names = FALSE)

In [ ]:
#== Identify AD prescription episodes that are monotherapy or allow short cross-tapers
#   - At START: ≤28-day overlap within first 28 days of episode
#   - At END: ≤28-day overlap that begins ≥28 days into episode AND within last 28 days

#-- Change to data table for this step to avoid kernel crashing
setDT(PrescriptionData_AD)
PrescriptionData_AD[, drug_exposure_start_date := as.Date(drug_exposure_start_date)]
PrescriptionData_AD[, final_end_date           := as.Date(final_end_date)]

#-- Unique episode id
PrescriptionData_AD[, episode_id := .I]

#-- Check size
print(nrow(PrescriptionData_AD))
print(length(unique(PrescriptionData_AD$person_id)))

#-- Split into patient chunks
all_ids <- unique(PrescriptionData_AD$person_id)
chunk_size <- 1000  
id_chunks <- split(
  all_ids,
  ceiling(seq_along(all_ids) / chunk_size)
)
length(id_chunks)

#========================
# mark combo vs monotherapy within one chunk
# - combo == TRUE  => disallowed overlap (long or mid-episode combination)
# - combo == FALSE => monotherapy OR only allowed early/late cross-taper
# - cross_taper_end_date: end of allowed START cross-taper (if any), else NA
#========================

mark_combo_chunk <- function(dt_chunk) {
  
  # Store original episode IDs
  original_episodes <- dt_chunk$episode_id
  
  setkey(dt_chunk, person_id, drug_exposure_start_date, final_end_date) 

  ov <- foverlaps(dt_chunk, dt_chunk, nomatch = NULL)
  
  # Get episodes with overlaps to OTHER drugs
  ov_other <- ov[(episode_id != i.episode_id) & (drug != i.drug)]
  
  # Create a fresh data.table with ALL original episodes
  result_dt <- copy(dt_chunk)
  result_dt[, combo := FALSE]
  result_dt[, cross_taper_end_date := as.Date(NA)]
  result_dt[, end_taper_start_date := as.Date(NA)]
  result_dt[, end_taper_drug := as.character(NA)]
  result_dt[, effective_episode_start := drug_exposure_start_date]
  result_dt[, effective_episode_end := final_end_date]

  if (nrow(ov_other) == 0L) {
    return(result_dt)
  }
  
  ov_other[, overlap_start := pmax(i.drug_exposure_start_date, drug_exposure_start_date)]
  ov_other[, overlap_end   := pmin(i.final_end_date, final_end_date)]
  ov_other[, overlap_days  := as.integer(overlap_end - overlap_start + 1)]
  
  # FIRST PASS: identify START tapers
  ov_other[, is_start_taper := 
             overlap_days <= 28 &
             overlap_start <= (i.drug_exposure_start_date + 28)]
  
  ov_other[, temp_allowed := is_start_taper]
  combo_ids_temp <- ov_other[temp_allowed == FALSE, unique(i.episode_id)]
  
  # Adjust for START tapers
  allowed_start_tapers <- ov_other[is_start_taper == TRUE & !(i.episode_id %in% combo_ids_temp)]
  
  if (nrow(allowed_start_tapers) > 0L) {
    taper_summary <- allowed_start_tapers[
      ,
      .(cross_taper_end_date = max(overlap_end, na.rm = TRUE)),
      by = i.episode_id
    ]
    setnames(taper_summary, "i.episode_id", "episode_id")
    result_dt[taper_summary, on = .(episode_id), 
             `:=`(cross_taper_end_date = i.cross_taper_end_date,
                  effective_episode_start = i.cross_taper_end_date + 1)]
  }
  
  # SECOND PASS: check END tapers using ADJUSTED start
  ov_other <- merge(ov_other, 
                    result_dt[, .(episode_id, effective_episode_start)], 
                    by.x = "i.episode_id", by.y = "episode_id", 
                    all.x = TRUE)
  
  ov_other[, is_end_taper := 
             overlap_days <= 28 &
             overlap_start >= (effective_episode_start + 28) &
             overlap_start >= (i.final_end_date - 28)]
  
  ov_other[, overlap_allowed := is_start_taper | is_end_taper]
  
  # Mark combo
  combo_ids <- ov_other[overlap_allowed == FALSE, unique(i.episode_id)]
  result_dt[episode_id %in% combo_ids, combo := TRUE]
  
  # Track END tapers
  allowed_end_tapers <- ov_other[is_end_taper == TRUE & !(i.episode_id %in% combo_ids)]
  
  if (nrow(allowed_end_tapers) > 0L) {
    end_taper_summary <- allowed_end_tapers[
      ,
      .(end_taper_start_date = min(overlap_start, na.rm = TRUE),
        end_taper_drug = drug[which.min(overlap_start)]),
      by = i.episode_id
    ]
    setnames(end_taper_summary, "i.episode_id", "episode_id")
    result_dt[end_taper_summary, on = .(episode_id), 
             `:=`(end_taper_start_date = i.end_taper_start_date,
                  end_taper_drug = i.end_taper_drug,
                  effective_episode_end = i.end_taper_start_date - 1)]
  }
  
  return(result_dt)
}

#== Loop over chunks and then combine
res_list <- vector("list", length(id_chunks))

for (k in seq_along(id_chunks)) {
  cat("Processing chunk", k, "of", length(id_chunks), "...\n")
  
  ids_k <- id_chunks[[k]]
  dt_k <- PrescriptionData_AD[person_id %in% ids_k]
  dt_k <- mark_combo_chunk(dt_k)
  
  res_list[[k]] <- dt_k
}

#-- Bind all chunks back together
PrescriptionData_AD_marked <- rbindlist(res_list)

#-- Calculate effective monotherapy duration and filter out negative/zero durations
PrescriptionData_AD_marked[, effective_monotherapy_duration := 
                             as.integer(effective_episode_end - effective_episode_start + 1)]

cat("Episodes before filtering:", nrow(PrescriptionData_AD_marked), "\n")
cat("Episodes with negative/zero duration:", 
    sum(PrescriptionData_AD_marked$effective_monotherapy_duration <= 0), "\n")

#-- Filter to keep only episodes with positive monotherapy duration
PrescriptionData_AD_marked <- PrescriptionData_AD_marked[effective_monotherapy_duration > 0]

cat("Episodes after filtering:", nrow(PrescriptionData_AD_marked), "\n")

#-- Final results
PrescriptionData_AD_mono  <- PrescriptionData_AD_marked[combo == FALSE]
PrescriptionData_AD_combo <- PrescriptionData_AD_marked[combo == TRUE]
PrescriptionData_AD_mono_df  <- as.data.frame(PrescriptionData_AD_mono)
PrescriptionData_AD_combo_df <- as.data.frame(PrescriptionData_AD_combo)

#-- Check how many recovered
cat("Monotherapy episodes:", nrow(PrescriptionData_AD_mono), "\n")
cat("Unique people with monotherapy:", length(unique(PrescriptionData_AD_mono$person_id)), "\n")
cat("Combo episodes:", nrow(PrescriptionData_AD_combo), "\n")
cat("Unique people with combo:", length(unique(PrescriptionData_AD_combo$person_id)), "\n")


In [ ]:
#-- Investigate number of monotherapy and combination therapy instances
length(unique(PrescriptionData_AD_combo_df$person_id))
length(unique(PrescriptionData_AD_mono_df$person_id))

#-- Apply minimum monotherapy duration requirement (≥28 days)
PrescriptionData_AD_mono[, effective_episode_duration := 
                           as.integer(effective_episode_end - effective_episode_start + 1)]

# Filter to episodes with ≥28 days of actual monotherapy
PrescriptionData_AD_mono_eligible <- PrescriptionData_AD_mono[effective_episode_duration >= 28]

cat("Total monotherapy episodes:", nrow(PrescriptionData_AD_mono), "\n")
cat("Total individuals with at least one monotherapy episode:", length(unique(PrescriptionData_AD_mono$person_id)), "\n")
cat("Eligible episodes (≥28 days):", nrow(PrescriptionData_AD_mono_eligible),"\n")
cat("Total individuals with at least one monotherapy episode of adequate length:", length(unique(PrescriptionData_AD_mono_eligible$person_id)), "\n")
cat("Excluded short episodes:", nrow(PrescriptionData_AD_mono) - nrow(PrescriptionData_AD_mono_eligible), "\n")

In [ ]:
#-- Adjust START but keep original END to capture cross-taper switches
PrescriptionData_AD_mono_eligible <- PrescriptionData_AD_mono_eligible %>%
  mutate(
    # Calculate true monotherapy duration for filtering
    effective_monotherapy_duration = as.integer(effective_episode_end - effective_episode_start + 1),
    
    # Adjust start date (exclude start cross-taper)
    drug_exposure_start_date = effective_episode_start
    
    # Keep original end date (INCLUDES end cross-taper period)
    # final_end_date stays as-is
  ) %>%
  select(-cross_taper_end_date, -episode_id, -combo, 
         -end_taper_drug,
         -effective_episode_start,
         -effective_monotherapy_duration, -end_taper_start_date)

In [ ]:
#== Main treatment outcomes
TreatmentContinuation=90

#-- Calculate Prescription Episode duration and label as 'continuation' or 'discontinuation'
PrescriptionEpisodes_mono <- PrescriptionData_AD_mono_eligible %>%
    group_by(person_id, drug, PrescriptionEpisode) %>%
    summarize(
      FirstDispense = min(drug_exposure_start_date),
      ExpectedEndDate = max(final_end_date),
      EpisodeDuration = as.numeric(difftime(max(effective_episode_end),  min(drug_exposure_start_date), units = "days")),
      EpisodeOutcome = ifelse(as.numeric(difftime(max(effective_episode_end), min(drug_exposure_start_date), units = "days")) >= TreatmentContinuation, 
                              "Continuation", "Discontinuation"),
      .groups = "drop"
    )
    
#-- Initial Classification
# First Dispense represents the first dispense of that drug from any prescription episode
# Expected End Date represents the expected end date of the last dispense of that drug from any prescription episode
InitialClassification <- PrescriptionEpisodes_mono %>%
    group_by(person_id, drug) %>%
    summarise(
      FirstDispense.treatment = min(FirstDispense),
      ExpectedEndDate.treatment = max(ExpectedEndDate),
      InitialClassification = ifelse(any(EpisodeOutcome == "Continuation"), "Continuation", "Discontinuation"),
      .groups = "drop"
    ) %>%
    rename(drug.treatment = drug)

#-- Identify switching behavior (only includes index monotherapy/short cross-taper prescription episodes 
# -- with adequate trial, i.e., at least 28 days ~ 4 weeks, the switched to episode caan be any AD episode)

#SwitchingFreeWindowLower = 0
#SwitchingFreeWindowUpper = 90
# The other prescription is a different antidepressant than the index treatment
# The new prescription starts at least 28 days after the index treatment began
# The new prescription starts no earlier than 28 days before the expected end of treatment
# The new prescription starts within 90 days after the expected end of treatment

SwitchingBehavior <- PrescriptionEpisodes_mono %>%
    inner_join(AD_PrescriptionEpisodes, by = "person_id", suffix = c(".treatment", ".other"), relationship = "many-to-many") %>%
    mutate(
      SwitchingClassification = case_when(
        drug.other != drug.treatment & 
          ((FirstDispense.other > FirstDispense.treatment + 28) & 
           (FirstDispense.other < FirstDispense.treatment + (6*30)) &
           (FirstDispense.other > ExpectedEndDate.treatment - 28) &
           (FirstDispense.other < ExpectedEndDate.treatment + 90))~ "Switching",
        TRUE ~ "Non-Switching"
      )
    ) 

#-- First Dispense represents the first dispense of that drug from any prescription episode
#-- Expected End Date represents the last dispense of that drug from any prescription episode
SwitchingClassification = SwitchingBehavior %>%
    group_by(person_id, drug.treatment) %>%
    arrange(FirstDispense.treatment) %>%
    summarize(
      FirstDispense.treatment = min(FirstDispense.treatment),
      ExpectedEndDate.treatment = max(ExpectedEndDate.treatment),
      InitialClassification = case_when(
        any(SwitchingClassification == "Switching") ~ "Switching",
        TRUE ~ "Non-Switching"),
      .groups = "drop"
    ) %>%
    arrange(person_id, ExpectedEndDate.treatment) %>%
    as.data.frame() %>%
    filter(InitialClassification == "Switching")

#-- Final Classifications
FinalClassifications <- rbind(SwitchingClassification, InitialClassification) %>%
    arrange(person_id, drug.treatment, FirstDispense.treatment, ExpectedEndDate.treatment) %>%
    group_by(person_id, drug.treatment) %>%
    summarize(
      FirstDispense.treatment = min(FirstDispense.treatment),
      ExpectedEndDate.treatment = max(ExpectedEndDate.treatment),
      FinalClassification.treatment = ifelse(any(InitialClassification == "Switching"), "Switching", InitialClassification),
      .groups = "drop"
    ) %>%
    arrange(person_id, FirstDispense.treatment, ExpectedEndDate.treatment) %>%
    as.data.frame()

#===== Populate data frame with index dates and respective prescription episodes ======

#-- Identify the date of earliest switch and to which drug for those with Switching
earliest_switch <- SwitchingBehavior %>%
    filter(SwitchingClassification == "Switching") %>%
    group_by(person_id, drug.treatment) %>%
    filter(FirstDispense.other == min(FirstDispense.other)) %>%
    filter(EpisodeDuration.other == max(EpisodeDuration.other)) %>%
    slice(1) %>%
    select(person_id, drug.treatment, 
           FirstDispense.treatment, ExpectedEndDate.treatment, EpisodeDuration.treatment,
           drug.other, FirstDispense.other, ExpectedEndDate.other, EpisodeDuration.other) %>%
    rename(
      FirstDispense.treatment.episode = FirstDispense.treatment,
      ExpectedEndDate.treatment.episode = ExpectedEndDate.treatment,
      EpisodeDuration.treatment.episode = EpisodeDuration.treatment,
      FirstDispense.other.episode = FirstDispense.other,
      ExpectedEndDate.other.episode = ExpectedEndDate.other,
      EpisodeDuration.other.episode = EpisodeDuration.other
    )


#-- Combine switching statistics with the Final Classifications data frame
FinalClassifications_switches <- left_join(FinalClassifications, earliest_switch, by = c("person_id", "drug.treatment")) %>%
    filter(FinalClassification.treatment == "Switching")


#-- Identify the latest initiation of the drugs that were "Discontinuation".
FinalClassifications_ED <- FinalClassifications %>%
    filter(FinalClassification.treatment == "Discontinuation")

earliest_discontinuation <- FinalClassifications_ED %>%
  left_join(PrescriptionEpisodes_mono, by = c("person_id", "drug.treatment" = "drug")) %>%
  group_by(person_id, drug.treatment) %>%
  filter(FirstDispense == min(FirstDispense)) %>%
  slice(1) %>%
  ungroup() %>%
  rename(
    FirstDispense.treatment.episode = FirstDispense,
    ExpectedEndDate.treatment.episode = ExpectedEndDate,
    EpisodeDuration.treatment.episode = EpisodeDuration
  ) %>%
  select(-EpisodeOutcome, -PrescriptionEpisode)

print(summary(earliest_discontinuation$EpisodeDuration.treatment.episode))

#-- Identify the earliest point of continuation classification for those with "Continuation"
FinalClassifications_C <- FinalClassifications %>%
    filter(FinalClassification.treatment == "Continuation")

earliest_continuation <- FinalClassifications_C %>%
    left_join(PrescriptionEpisodes_mono, by = c("person_id", "drug.treatment" = "drug")) %>%
    group_by(person_id, drug.treatment) %>%
    filter(EpisodeOutcome == "Continuation") %>%
    filter(FirstDispense == min(FirstDispense)) %>%
    slice(1) %>%
    ungroup() %>%
    rename(
      FirstDispense.treatment.episode = FirstDispense,
      ExpectedEndDate.treatment.episode = ExpectedEndDate,
      EpisodeDuration.treatment.episode = EpisodeDuration
    ) %>%
    select(-EpisodeOutcome, -PrescriptionEpisode)

print(summary(earliest_continuation$EpisodeDuration.treatment.episode))


#== Combine mutually exclusive classification dataframes
FinalClassifications_all <- bind_rows(FinalClassifications_switches, earliest_discontinuation, earliest_continuation)

#-- Define Incidence date for each major outcome: Early Discontinuation, Continuation, and Switching
FinalClassifications_index <- FinalClassifications_all %>%
    mutate(Incidence = case_when(
      FinalClassification.treatment == "Continuation" ~ as.Date(FirstDispense.treatment.episode) + TreatmentContinuation,
      FinalClassification.treatment == "Discontinuation" ~ as.Date(ExpectedEndDate.treatment.episode),
      FinalClassification.treatment == "Switching" ~ as.Date(FirstDispense.other.episode),
      TRUE ~ NA
    ),
    # Define index date for switching and non-switching
    Index = case_when(
        FinalClassification.treatment == "Switching" ~ as.Date(FirstDispense.treatment.episode),
        FinalClassification.treatment %in% c("Discontinuation", "Continuation") ~ as.Date(FirstDispense.treatment),
        TRUE ~ NA)
    )

#== Identify the expected end date of first monotherapy episode for those who are non-switchers
earliest_monotherapy <- FinalClassifications %>%
    filter(FinalClassification.treatment %in% c("Continuation", "Discontinuation")) %>%
    left_join(PrescriptionEpisodes_mono, by = c("person_id", "drug.treatment" = "drug")) %>%
    group_by(person_id, drug.treatment) %>%
    filter(FirstDispense == min(FirstDispense)) %>%
    slice(1) %>%
    ungroup() %>%
    rename(
      FirstDispense.first.episode = FirstDispense,
      ExpectedEndDate.first.episode = ExpectedEndDate,
      EpisodeDuration.first.episode = EpisodeDuration
    ) %>%
    select(person_id, drug.treatment, FirstDispense.first.episode, ExpectedEndDate.first.episode, EpisodeDuration.first.episode)

#== Define concomitant CYP exposure window
FinalClassifications_index <- FinalClassifications_index %>%
  left_join(earliest_monotherapy, by = c("person_id", "drug.treatment")) %>%
  mutate(
    cyp_concomitant_upper_window =
      case_when(
        # Non-switchers: index -> min( index+90, end of first monotherapy episode )
        FinalClassification.treatment != "Switching" ~
          pmin(Index + 90, ExpectedEndDate.first.episode, na.rm = TRUE),

        # Switchers: index -> min( switch date (Incidence), index+90)
        FinalClassification.treatment == "Switching" ~
          pmin(Incidence, Index + 90, na.rm = TRUE)
      )
  )

#== Add in date of first recorded prescription dispense 
Final <- left_join(FinalClassifications_index, info, by = c("person_id", "drug.treatment" = "drug"))

#-- Add in category
categories <- PrescriptionData_AD_mono %>%
  select(drug, Category) %>% 
  distinct()
Final_Category <- left_join(left_join(Final, categories, by = c("drug.treatment" = "drug")), categories, by = c("drug.other" = "drug"))
Final_Category <-  Final_Category %>%
    rename(
    Category.treatment = Category.x,
    Category.other = Category.y)

# duration of last episode of discontinuation
# duration of first episode of continuation

In [ ]:
#== Add in general adherence scores prior to incidence
PrescriptionData_adherence_select <- PrescriptionData_adherence %>%
    select(person_id, drug, drug_exposure_start_date, PrevExpectedEndDate, Adherent)

GA_prior <- Final_Category %>%
    left_join(PrescriptionData_adherence_select, by = c("person_id"), relationship = "many-to-many") %>%
    filter(drug_exposure_start_date <= FirstDispense.treatment.episode)
GA_prior_scores <- GA_prior %>%
    mutate(Adherent = replace_na(Adherent, 0)) %>%
    group_by(person_id, drug.treatment) %>%
    mutate(
    General_adherence_score.prior = mean(Adherent)
    ) %>% 
    ungroup() %>%
    select(person_id, drug.treatment, General_adherence_score.prior ) %>%
    distinct()
    
ATC_GA_prior <- Final_Category %>%
    left_join(PrescriptionData_adherence_select, by = c("person_id", "drug.treatment" = "drug"), relationship = "many-to-many") %>%
    filter(drug_exposure_start_date <= FirstDispense.treatment.episode)
ATC_GA_prior_scores <- ATC_GA_prior %>%
    mutate(Adherent = replace_na(Adherent, 0)) %>%
    group_by(person_id, drug.treatment) %>%
    mutate(
    ATC_adherence_score.prior = mean(Adherent)
    ) %>% 
    ungroup() %>%
    select(person_id, drug.treatment, ATC_adherence_score.prior ) %>%
    distinct()

Final_GA <- Final_Category %>%
    left_join(GA_prior_scores, by = c("person_id", "drug.treatment")) %>%
    left_join(ATC_GA_prior_scores, by = c("person_id", "drug.treatment"))

In [ ]:
#-- Save results for first analyses. Participants can have more than one classification.
write.csv(Final_GA, paste0("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_", TreatmentContinuation, "days_sens.csv"), row.names = FALSE, quote = FALSE)

In [ ]:
#-- Exclude those with SCZ and BIP from the study completely
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")
psychotic <- pheno %>% filter(Psychotic == 1)
bip <- pheno %>% filter(BIP==1)
Final_GA_noPsychotic_BIP <- Final_GA %>%
    filter(!person_id %in% psychotic$person_id) %>%
    filter(!person_id %in% bip$person_id)
length(unique(Final_GA_noPsychotic_BIP$person_id)); dim(Final_GA_noPsychotic_BIP)
write.csv(Final_GA_noPsychotic_BIP, paste0("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_", TreatmentContinuation, "days_noBIP_noPsychotic_sens.csv"), row.names = FALSE, quote = FALSE)